# 🤖 Project 13.4 — Robot Mission Planner
**DSA for Mechatronics · Week 13 — Greedy Algorithms**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, heapq
from collections import defaultdict
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A robot must plan missions efficiently: find the shortest path to a target,
load the most valuable cargo within its weight limit, and schedule tasks
before their deadlines.

1. **Dijkstra's shortest path** — greedy: always relax the unvisited node with
   the smallest current distance (min-heap implementation)
2. **Fractional knapsack** — greedy: sort items by value/weight ratio; take
   as much as possible of the best items first
3. **Deadline task scheduling** — greedy: schedule tasks to maximise profit
   before deadlines (sort by profit descending, place in latest available slot)


## Step 1 — Generate graph, cargo, and task datasets

In [ ]:
# ── Personalised parameters ──────────────────────────────────────
N_NODES   = random.randint(7, 11)
EDGE_P    = random.uniform(0.40, 0.60)
W_MAX_D   = random.randint(15, 30)
N_CARGO   = random.randint(6, 10)
CAPACITY  = random.randint(30, 50)
N_TASKS   = random.randint(8, 12)
MAX_DL    = random.randint(4, 6)

# Dijkstra graph (directed, non-negative weights)
NODES_D   = list(range(N_NODES))
EDGES_D   = []
# Guaranteed path: 0 → 1 → 2 → ... → N-1
for i in range(N_NODES - 1):
    w = random.randint(1, W_MAX_D)
    EDGES_D.append((i, i+1, w))
for u in range(N_NODES):
    for v in range(N_NODES):
        if u != v and random.random() < EDGE_P:
            if not any(a==u and b==v for a, b, _ in EDGES_D):
                EDGES_D.append((u, v, random.randint(1, W_MAX_D)))
ADJ_D = defaultdict(list)
for u, v, w in EDGES_D:
    ADJ_D[u].append((w, v))

# Cargo items: (name, weight, value)
CARGO = [(f"C{i+1}", round(random.uniform(2, 15), 1),
          round(random.uniform(10, 60), 1)) for i in range(N_CARGO)]

# Tasks: (name, deadline_slot, profit)
TASKS = [(f"T{i+1}", random.randint(1, MAX_DL), random.randint(10, 80))
         for i in range(N_TASKS)]

SOURCE = 0
TARGET = N_NODES - 1

print(f"Robot mission parameters:")
print(f"  Graph  : {N_NODES} nodes, {len(EDGES_D)} directed edges")
print(f"  Source : {SOURCE}   Target : {TARGET}")
print()
print(f"  Cargo items (n={N_CARGO}):")
print(f"  {'Name':<6}  {'Weight':>8}  {'Value':>8}  {'V/W ratio':>10}")
print(f"  {'─'*6}  {'─'*8}  {'─'*8}  {'─'*10}")
for name, w, v in CARGO:
    print(f"  {name:<6}  {w:>8.1f}  {v:>8.1f}  {v/w:>10.3f}")
print()
print(f"  Tasks (n={N_TASKS}, capacity={MAX_DL} time slots):")
print(f"  {'Name':<6}  {'Deadline':>9}  {'Profit':>8}")
print(f"  {'─'*6}  {'─'*9}  {'─'*8}")
for name, dl, profit in sorted(TASKS, key=lambda x: -x[2]):
    print(f"  {name:<6}  {dl:>9}  {profit:>8}")


## Step 2 — Dijkstra's shortest path

In [ ]:
def dijkstra(n_nodes, adj, source, target):
    """
    Dijkstra's algorithm: find shortest path from source to target.

    Greedy strategy: always relax the unvisited node with the
    smallest tentative distance.
    Min-heap ensures we always process the globally closest node next.

    Why greedy works: once a node is popped from the heap, its distance
    is final (all edge weights ≥ 0 — negative weights break this invariant).

    dist[v] = shortest known distance from source to v.
    prev[v] = predecessor on shortest path (for path reconstruction).
    O((V + E) log V) with a min-heap.
    """
    dist    = {v: float('inf') for v in range(n_nodes)}
    prev    = {v: None         for v in range(n_nodes)}
    dist[source] = 0
    heap    = [(0, source)]
    visited = set()
    steps   = []

    while heap:
        d, u = heapq.heappop(heap)
        if u in visited: continue
        visited.add(u)
        steps.append((u, d))
        if u == target: break
        for w, v in adj[u]:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                prev[v] = u
                heapq.heappush(heap, (dist[v], v))

    # Reconstruct path
    path = []
    node = target
    while node is not None:
        path.append(node)
        node = prev[node]
    path.reverse()
    reachable = dist[target] < float('inf')
    return dist[target] if reachable else -1, path if reachable else [], steps

sp_dist, sp_path, dijk_steps = dijkstra(N_NODES, ADJ_D, SOURCE, TARGET)

print(f"Dijkstra's shortest path (source={SOURCE} → target={TARGET}):")
print()
print(f"  Edge list (directed):")
for u, v, w in sorted(EDGES_D, key=lambda x: x[2]):
    print(f"    {u}→{v}  w={w}")
print()
print(f"  Dijkstra expansion order:")
for node, dist_val in dijk_steps:
    on_path = node in sp_path
    print(f"    settle node {node}  dist={dist_val}  {'← on SP' if on_path else ''}")
print()
if sp_dist >= 0:
    print(f"  Shortest path    : {' → '.join(map(str, sp_path))}")
    path_weights = []
    for i in range(len(sp_path)-1):
        u, v = sp_path[i], sp_path[i+1]
        w = next(w for a, b, w in EDGES_D if a==u and b==v)
        path_weights.append(w)
    print(f"  Edge weights     : {path_weights}")
    print(f"  Total distance   : {sp_dist}")
    verify = sum(path_weights) == sp_dist
    print(f"  Sum verify       : {sum(path_weights)} == {sp_dist}  {'✅' if verify else '❌'}")
else:
    print(f"  No path exists from {SOURCE} to {TARGET}")
    verify = True


## Step 3 — Fractional knapsack

In [ ]:
def fractional_knapsack(items, capacity):
    """
    Fractional Knapsack: maximise value with limited weight; items are divisible.

    Greedy strategy: sort items by VALUE/WEIGHT ratio (density) descending.
    Take as much as possible of the highest-density item first.
    If an item doesn't fit entirely, take the fraction that fills remaining space.

    Why greedy works (exchange argument): if we ever leave some of a
    high-density item for a low-density one, swapping increases total value.
    This proof FAILS for 0/1 knapsack (where items are indivisible).

    O(n log n) for sort + O(n) scan.
    """
    # Sort by value/weight ratio descending
    sorted_items = sorted(items, key=lambda x: x[2]/x[1], reverse=True)
    total_value  = 0.0
    remaining    = capacity
    taken        = []
    for name, w, v in sorted_items:
        if remaining <= 0: break
        frac = min(1.0, remaining / w)
        taken.append((name, w, v, frac, frac*v))
        total_value += frac * v
        remaining   -= frac * w
    return total_value, taken

fk_value, fk_taken = fractional_knapsack(CARGO, CAPACITY)

print(f"Fractional knapsack (capacity={CAPACITY}):")
print()
print(f"  {'Name':<6}  {'Weight':>8}  {'Value':>8}  {'V/W':>8}  {'Fraction':>10}  {'Value taken':>12}")
print(f"  {'─'*6}  {'─'*8}  {'─'*8}  {'─'*8}  {'─'*10}  {'─'*12}")
for name, w, v, frac, val_taken in fk_taken:
    print(f"  {name:<6}  {w:>8.1f}  {v:>8.1f}  {v/w:>8.3f}  "
          f"{frac:>10.3f}  {val_taken:>12.2f}")
print()
weight_used = sum(w * frac for _, w, _, frac, _ in fk_taken)
print(f"  Weight used      : {weight_used:.2f} / {CAPACITY}")
print(f"  Total value      : {fk_value:.2f}")

# Compare with 0/1 DP knapsack
import math
def dp_knapsack_int(items, cap):
    """0/1 knapsack with integer weights (multiply by 10 for 1 decimal)."""
    cap_i = int(cap * 10)
    items_i = [(name, int(w*10), v) for name, w, v in items]
    dp = [0.0] * (cap_i + 1)
    for name, wi, vi in items_i:
        for c in range(cap_i, wi - 1, -1):
            dp[c] = max(dp[c], dp[c - wi] + vi)
    return dp[cap_i]

dp_01_value = dp_knapsack_int(CARGO, CAPACITY)
print(f"  0/1 DP knapsack  : {dp_01_value:.2f}")
print(f"  Fractional ≥ 0/1 : {'✅ YES' if fk_value >= dp_01_value - 0.01 else '❌ NO (unexpected)'}")
print(f"  Extra from dividing: {fk_value - dp_01_value:.2f}")


## Step 4 — Deadline task scheduling (maximise profit)

In [ ]:
def deadline_schedule(tasks, max_deadline):
    """
    Task scheduling with deadlines: select tasks to maximise total profit.
    Each task takes 1 time unit and must complete by its deadline.

    Greedy strategy: sort by PROFIT descending.
    For each task (highest profit first), place it in the latest available
    slot at or before its deadline.

    This greedy works because:
    - We always consider the highest-value tasks first.
    - Placing each task as LATE as possible leaves earlier slots free
      for tasks with tighter deadlines.

    Implementation: slot array of size max_deadline.
    O(n log n) sort + O(n * max_deadline) placement (small constant for
    typical scheduling problems; can be O(n log n) with union-find).
    """
    sorted_tasks = sorted(tasks, key=lambda x: -x[2])   # sort by profit desc
    slots = [None] * (max_deadline + 1)   # slots[1..max_deadline]
    selected = []
    total_profit = 0
    for name, dl, profit in sorted_tasks:
        # Find latest free slot ≤ deadline
        placed = False
        for t in range(min(dl, max_deadline), 0, -1):
            if slots[t] is None:
                slots[t] = name
                selected.append((t, name, dl, profit))
                total_profit += profit
                placed = True
                break
    selected.sort()   # sort by slot for display
    return selected, total_profit, slots

sched, sched_profit, slots = deadline_schedule(TASKS, MAX_DL)

print(f"Deadline task scheduling (n={N_TASKS}, {MAX_DL} time slots):")
print()
print(f"  All tasks (sorted by profit):")
print(f"  {'Name':<6}  {'Deadline':>9}  {'Profit':>8}  {'Status':>10}")
print(f"  {'─'*6}  {'─'*9}  {'─'*8}  {'─'*10}")
sched_names = {name for _, name, _, _ in sched}
for name, dl, profit in sorted(TASKS, key=lambda x: -x[2]):
    status = "✅ scheduled" if name in sched_names else "  skipped  "
    print(f"  {name:<6}  {dl:>9}  {profit:>8}  {status}")
print()
print(f"  Schedule (slot → task):")
for t in range(1, MAX_DL + 1):
    task_name = slots[t]
    if task_name:
        task_info = next((n, dl, p) for n, dl, p in TASKS if n == task_name)
        print(f"    Slot {t}: {task_name}  (deadline={task_info[1]}, profit={task_info[2]})")
    else:
        print(f"    Slot {t}: [empty]")
print()
print(f"  Tasks scheduled  : {len(sched)} of {N_TASKS}")
print(f"  Total profit     : {sched_profit}")
total_all = sum(p for _, _, p in TASKS)
print(f"  Max possible     : {total_all}  (if no deadlines)")
print(f"  Efficiency       : {sched_profit/total_all:.1%}")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " ROBOT MISSION PLANNER — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  DIJKSTRA SHORTEST PATH" + " "*(W-24) + "║")
print(f"║  {'Nodes':<28}: {N_NODES:<26}║")
print(f"║  {'Edges (directed)':<28}: {len(EDGES_D):<26}║")
if sp_dist >= 0:
    print(f"║  {'Shortest distance':<28}: {sp_dist:<26}║")
    print(f"║  {'Path':<28}: {' → '.join(map(str, sp_path)):<26}║")
else:
    print(f"║  {'Shortest distance':<28}: {'No path exists':<26}║")
print("╠" + "═"*W + "╣")
print("║  FRACTIONAL KNAPSACK" + " "*(W-21) + "║")
print(f"║  {'Cargo items':<28}: {N_CARGO:<26}║")
print(f"║  {'Weight capacity':<28}: {CAPACITY:<26}║")
print(f"║  {'Fractional value':<28}: {fk_value:.2f}{'':<22}║")
print(f"║  {'0/1 DP value':<28}: {dp_01_value:.2f}{'':<22}║")
print(f"║  {'Fractional ≥ 0/1':<28}: {'YES ✅' if fk_value >= dp_01_value - 0.01 else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  DEADLINE TASK SCHEDULING" + " "*(W-26) + "║")
print(f"║  {'Tasks (n)':<28}: {N_TASKS:<26}║")
print(f"║  {'Time slots':<28}: {MAX_DL:<26}║")
print(f"║  {'Tasks scheduled':<28}: {len(sched)} of {N_TASKS}{'':<19}║")
print(f"║  {'Total profit':<28}: {sched_profit:<26}║")
print(f"║  {'Max possible (no deadlines)':<28}: {total_all:<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about greedy algorithms in this context?

*Your answer here:*

---

**Q2 — Why does the greedy choice work here?**
For one of the algorithms in this project, explain the *exchange argument* — why swapping the greedy choice for any other choice cannot improve the solution.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
